# Notizen

- Vorteile Polars
https://docs.pola.rs/user-guide/migration/pandas/
  - schneller da konsequent in Rust implementiert 
  - speicherärmer: nutzt nativ Apache Pyarrow Datentypen 
  - mehr Datentypen, z.B. auch Timedelta, struct, list
  - Null Datentyp für fehlende Daten, in allen Datentypen vorhanden
  - kompatibel mit Delta Format
  - Sowohl Lazy als auch Greedy computing
  - bessere Unterstützung für Parallelisierung
- weitere Unterschiede
  - kein Index in polars Dataframes, kann via Befehl eingefügt werden
  - mehr SQL-like, d.h. formal strenger

- Typische Befehle
  - Import: `import polars as pl`
  - Einlesen Datei: `df = pl.read_parquet("data.par")`
  - Umwandlung nach Pandas df: `df_pandas = df_polars.to_pandas()`
  - Spalten selektieren
    - Spalte als Series: `df["a"]`
    - Bestimmte Spalten: `df.select(["A","B"])` oder `df.select(pl.col("A"),pl.col("B"))`
    - Bestimmte Datentypen: `df.select(pl.selectors.String())`
  - Zeilen selektieren
    - Filtern: `df.filter(pl.col("A").is_in([1,2,3]))`
    - Entfernen: `df.remove(pl.col("A").is_null)`

# Imports

In [ ]:
df = (
    df.select(...)
    .filter(..)
    .sort(..)
)

df = df.select()
df = df.filter

In [ ]:
import pathlib,datetime,io,copy,time,re,json,itertools,string,hashlib
#
import numpy as np
RNG = np.random.default_rng(seed=42)
import pandas as pd
import polars as pl
pl.Config.set_tbl_rows(99)
import matplotlib.pyplot as plt
plt.rcParams['axes.grid']=True
plt.rcParams['axes.axisbelow'] = True
#
print(f"{np.__version__=}")
print(f"{pl.__version__=}")

# Datensatzgröße

In [ ]:
N = 1_000_000
C = 100
#
Data_float  = RNG.random(size=(N,C),dtype=np.float32)
Data_int    = RNG.integers(low=0,high=1000,size=(N,C),dtype=np.int32)

In [ ]:
df_float = pd.DataFrame(Data_float)
df_int   = pd.DataFrame(Data_int)

In [ ]:
dg_float = pl.DataFrame(Data_float)
dg_int   = pl.DataFrame(Data_int)

In [ ]:
(
    df_float.memory_usage(deep=True).sum()>>20,
    df_int.memory_usage(deep=True).sum()>>20
)

In [ ]:
(
    dg_float.estimated_size()>>20,
    dg_int.estimated_size()>>20
)

In [ ]:
df_float.to_parquet("df_float.par")
df_int.to_parquet("df_int.par")

In [ ]:
dg_float.write_parquet("dg_float.par")
dg_int.write_parquet("dg_int.par")

In [ ]:
for file in pathlib.Path().glob("*.par"):
    print(f"{file.name:<15} : {file.stat().st_size>>20:,d} MB")

# Performance

## Kreuzprodukt

In [ ]:
A = list(string.ascii_lowercase)
B = list(string.ascii_uppercase)
C = range(50000)
#
print(f"{len(A)*len(B)*len(C):_}")

In [ ]:
# pandas
dp = pd.DataFrame({'A': A })
#
dp = dp.join(pd.DataFrame({'B': B }), 
             how="cross")
#
dp = dp.join(pd.DataFrame({'C': C }), 
             how="cross")

In [ ]:
dp.head()

In [ ]:
# polars
df = pl.DataFrame({'A': A })
#
df = df.join(pl.DataFrame({'B': B }), 
             how="cross")
#
df = df.join(pl.DataFrame({'C': C }), 
             how="cross")

In [ ]:
df.head()

In [ ]:
X = dp.head(1000)
Y = df.head(1000)
print("Pandas:",dp.shape,dp.__sizeof__()>>20)
print("Polars:",df.shape,df.estimated_size()>>20)
print(Y.equals(pl.from_pandas(X)))

# Kontingenztafel

In [ ]:
n = 1_000_000_0
a = np.random.randint(0,10,n)
b = np.random.randint(10,20,n)
c = np.random.choice(list(string.ascii_lowercase)[:10],n)
d = np.random.choice(list(string.ascii_uppercase)[:10],n)

In [ ]:
df = pl.DataFrame({'c1':a,'c2':b,'c3':c,'c4':d})
df.head()

In [ ]:
# Pivotisieren von c1 vs. c2
# Polars = df.pivot(on="c1",index="c2",values="c1",aggregate_function="len").sort('c2')
Polars = df.pivot(on="c1",index="c2",values="c1",aggregate_function="len").sort('c2')
Polars

In [ ]:
%%timeit
# Geschw.-Test
df.pivot(on="c1",index="c2",values="c1",aggregate_function="len").sort('c2')

In [ ]:
df = pd.DataFrame({'c1':a,'c2':b,'c3':c,'c4':d})
df.head()

In [ ]:
Pandas = pd.crosstab(df['c1'],df['c2'])

In [ ]:
%%timeit
pd.crosstab(df['c1'],df['c2'])

In [ ]:
%%timeit
pl.from_pandas(pd.crosstab(df['c1'],df['c2']),include_index=True)

In [ ]:
%%timeit
df.pivot(on="c3",index="c4",values="c3",aggregate_function="len").sort('c4')

In [ ]:
%%timeit
pl.from_pandas(pd.crosstab(df['c3'],df['c4']),include_index=True)

In [ ]:
%%timeit
df.pivot(on="c1",index="c4",values="c1",aggregate_function="len").sort('c4')

In [ ]:
%%timeit
pl.from_pandas(pd.crosstab(df['c1'],df['c4']),include_index=True)